# 02 · Recoverable Qwen3.5 Bengali LoRA continued pretraining

The objective is next-token prediction on authentic literary passages. The official
`Qwen/Qwen3.5-0.8B-Base` checkpoint includes a vision tower; the loader extracts and
verifies the text backbone using `Qwen3_5ForCausalLM`, never silently initializes missing
language weights, freezes base parameters, and checks every requested LoRA target.

**All training and model downloads are disabled by default.** This repository was
requested without executing training. The memory probe, full-model passes and recovery
demonstration below are real implementations, with no fabricated results.


In [ ]:
from pathlib import Path
import sys, json, logging

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "src/tagore_llm").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
from tagore_llm.common import load_config, read_json, read_jsonl, write_json
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
PROJECT = load_config(ROOT)
print("Project:", ROOT)
print("Pinned model:", PROJECT["model_id"], "revision:", PROJECT["revision"])


In [ ]:
TRAINING = load_config(ROOT, "training")
CHECK_MODEL_AND_TOKENIZER = False
RUN_DATASET_AUDIT = False
RUN_TRAINING = False
RUN_RECOVERY_DEMO = False
print(json.dumps(TRAINING, indent=2))


## Hardware and dependencies

Use Python 3.11 or 3.12 on Linux/WSL2. Install the CUDA-enabled PyTorch wheel before
`requirements.txt`. `nvidia-smi` reports driver-supported CUDA; `torch.version.cuda`
reports the installed wheel's runtime. Verify the actual laptop's power limit instead
of assuming 95 W. GPU availability alone does not prove a 512-token training run fits.


In [ ]:
from tagore_llm.modeling import environment_report
try:
    hardware = environment_report(ROOT)
    print(json.dumps(hardware, indent=2))
except ModuleNotFoundError as exc:
    print("Dependencies are not installed:", exc)
    print("Follow README installation instructions before enabling training.")


## Conservative configuration

Start with one epoch, length 512, batch 1, accumulation 8, rank 8/alpha 16/dropout 0,
learning rate 2e-5, cosine schedule, warmup 0.03, weight decay 0.01 and clipping 1.0.
BF16 is selected only if PyTorch confirms support; otherwise FP16 with framework scaling.
Gradient checkpointing uses the non-reentrant implementation; cache is disabled for training.

The baseline uses Transformers Trainer plus PEFT, with the pinned TRL package available
for controlled experiments. Direct pretokenized causal labels avoid additional SFT
formatting or packing transformations. Unsloth officially supports Qwen3.5, but is not
mixed into this baseline: its patched stack needs a separate verified environment.
Unsloth currently discourages 4-bit QLoRA for Qwen3.5. The 8-bit AdamW **optimizer**
is separate from quantizing base weights.


In [ ]:
if CHECK_MODEL_AND_TOKENIZER:
    from tagore_llm.modeling import verify_model
    from tagore_llm.common import tokenizer_for, tokenizer_fingerprint
    _, model_report = verify_model(PROJECT)
    tokenizer = tokenizer_for(PROJECT)
    print(json.dumps(model_report, indent=2))
    print("Tokenizer fingerprint:", tokenizer_fingerprint(tokenizer))
else:
    print("Model/tokenizer download and inspection disabled.")

if RUN_DATASET_AUDIT:
    from tagore_llm.common import tokenizer_for
    from tagore_llm.corpus import validate_dataset
    tokenizer = tokenizer_for(PROJECT)
    print(validate_dataset(ROOT, PROJECT, tokenizer))


## Full-state checkpoint recovery

Every 25 optimizer steps, Trainer saves adapter weights, optimizer, scheduler, global
step, epoch, trainer configuration and RNG states (plus scaler state when used). After
file verification and fsync, this project writes `COMPLETE.json` with SHA-256 checksums.
Rotation keeps three **complete** saves. A partial newest folder cannot displace a
recoverable earlier save. Old incomplete folders remain visible for diagnosis.

Relaunch this same notebook or CLI with the same configs and run_name: the latest
complete compatible checkpoint is selected and passed to `Trainer.train` through
`resume_from_checkpoint`. Incompatible model/tokenizer/dataset/config/code/package
fingerprints raise an error. `force_fresh=true` creates a new run directory.

Save to a persistent SSD; `checkpoint_root` accepts an absolute path. Unsaved steps
since the latest successful save can be lost after power failure or a hard crash.
One SIGINT/SIGTERM requests a save after the current optimizer step. A second interrupt,
SIGKILL, closing an unprotected terminal, or disk failure can interrupt the current save.


In [ ]:
from tagore_llm.checkpoints import checkpoint_inventory
checkpoint_root = Path(TRAINING["checkpoint_root"]).expanduser()
if not checkpoint_root.is_absolute():
    checkpoint_root = ROOT / checkpoint_root
run_dir = checkpoint_root / TRAINING["run_name"]
if (run_dir / "RUN.json").exists():
    inventory = checkpoint_inventory(run_dir, read_json(run_dir / "RUN.json"))
    print(json.dumps(inventory, indent=2))
else:
    print("No existing training run at", run_dir)


## Train with a measured forward/backward memory probe

Enabling the cell below loads the real model, rechecks data hashes/token counts/leakage,
applies LoRA, and probes the longest prepared sequence before training. The probe
reports measured peak allocated/reserved memory and estimates optimizer headroom.
It is a gate, not a guarantee for all subsequent steps. CPU Arrow data stays off CUDA.

Loss, learning rate, gradient norm, actual non-padding training tokens/s, elapsed session
time, epoch, optimizer step, GPU memory and measured ETA are logged in `logs/<run>/`.
Held-out NLL is also evaluated with exact target-token weighting. Validation-gap and
rising-loss warnings inform the decision to stop; a second epoch requires a new run.
Optional early stopping is configured by `early_stopping_patience` (>0).


In [ ]:
if RUN_TRAINING:
    from tagore_llm.training import train
    result = train(ROOT, PROJECT, TRAINING)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("Training disabled. No weights loaded and no optimizer steps executed.")


## Real stop / process exit / resume demonstration (explicit opt-in)

This runs **four real optimizer steps** on your prepared corpus in two separate Python
processes, with one fixed four-step scheduler horizon. The first stops at step 2 and
saves; the second restores full state and reaches step 4. It verifies adapter changes,
optimizer step counts, scheduler progression and checkpoint seals, then loads the
exported adapter and saves a short Bengali-prompt base/adapter comparison. It does not
create fake literary examples or infer success from folder names alone.

The demonstration uses a unique experiment name. It is not the main training run.
**It has not been executed as part of creating this repository.**


In [ ]:
if RUN_RECOVERY_DEMO:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "scripts/recovery_demo.py"), "--execute"], cwd=ROOT, check=True)
else:
    print("Recovery demonstration not executed.")


## OOM, disk errors, and the next epoch decision

On OOM, restart the kernel before loading the model again. For an unchanged configuration,
resume the latest valid save. To reduce sequence length, set 256 in **both** configs,
rebuild/audit the dataset, and choose a new run_name; try 128 if still necessary. A changed
dataset/scheduler/batch layout must not reuse incompatible optimizer state. Keep batch 1
and dataloader workers 0. If bitsandbytes fails, choose `optimizer="adamw_torch"` in a new
experiment and rerun the memory probe. Do not assume 4-bit quantization is a safe fix.

A short run may not reach eval_steps=50; final evaluation and a full checkpoint still run.
Inspect held-out loss, generation comparisons, originality flags and human ratings before
planning another epoch. A lower training loss alone is not evidence of better writing.
See README for terminal persistence, relaunch commands, export paths and disk planning.
